# Synthetic lateral-movement sensitivity experiment

This notebook adds sparse aggregate-flow overlays to NF-CSE-CIC-IDS2018 Day 2 and evaluates the existing champion checkpoints without retraining or threshold adjustment. It creates **synthetic NetFlow-like records, not packets**, and does not prove successful authentication or remote execution.

The fixed experiment contains 54 attack scenarios: SMB/RPC, RDP, and SSH × three discovered targets × 5/15/30-minute horizons × direct valid-account or authentication-attempt paths. Nine endpoint-permuted benign controls test whether responses are caused only by service-flow features.

In [ ]:
# Colab setup
from pathlib import Path
import sys

from google.colab import drive
drive.mount('/content/drive')

!pip -q install torch-geometric

REPO_ROOT = Path('/content/temporalgnn-nids')
if not REPO_ROOT.exists():
    raise FileNotFoundError(
        'Clone or upload temporalgnn-nids to /content/temporalgnn-nids first.'
    )
sys.path.insert(0, str(REPO_ROOT / 'code/python'))

In [ ]:
# Paths and immutable experiment settings
import torch

ANALYSIS_ROOT = Path('/content/drive/MyDrive/nids-mitre')
DAY2_CSV = ANALYSIS_ROOT / 'data/cicids2018-v3/cicids2018v3_thu0103.csv'
DAY2_GRAPH_ROOT = ANALYSIS_ROOT / 'dataset_processed_thu0103'
RECOVERED_MAP = DAY2_GRAPH_ROOT / 'ip_map_day2_recovered.pkl'
SCALER_PATH = ANALYSIS_ROOT / 'dataset_processed/scaler_wed2802.pkl'
RESULTS_DIR = ANALYSIS_ROOT / 'results_earlystopping'
CORRECTED_CAMPAIGN_DIR = ANALYSIS_ROOT / 'analysis_mitre_corrected_v1'
OUTPUT_DIR = ANALYSIS_ROOT / 'synthetic_lateral_movement_v1'

PIVOT_IP = '172.31.69.13'
ATTACKER_IP = '13.58.225.34'
GENERATION_SEED = 20260804
OVERWRITE_OVERLAYS = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)
print('Output:', OUTPUT_DIR)

In [ ]:
# Imports
import gc
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.evaluation import gather_metrics, apply_1sd_rule
from utils.models import (
    SimpleMLP, E_GraphSAGE, EdgeGRU_Baseline_NoX,
    StaticGNN_Identity, ST_GNN_Identity,
)
from utils.synthetic_lm import (
    SyntheticLMError, apply_diagnostic_gate, attach_corrected_campaign_ground_truth,
    audit_ip_map_usage, build_raw_timing_index, build_scenario_matrix,
    create_matched_controls, create_synthetic_flows, donor_support_report,
    enrich_base_availability, evaluate_model_scenarios, load_ip_map,
    load_relevant_day2_flows, select_targets, summarize_experiment,
    validate_overlay_layout, validate_synthetic_flows, write_sparse_overlays,
)

## 1. Preflight and scenario construction

Targets must have exact-port discovery evidence from the infected host. Donor support must contain either at least 20 exact-port internal TCP flows or at least 50 flows in the same model port category. Failure stops the experiment rather than inventing unsupported values.

In [ ]:
required_paths = [DAY2_CSV, DAY2_GRAPH_ROOT / 'test2', RECOVERED_MAP, SCALER_PATH, RESULTS_DIR, CORRECTED_CAMPAIGN_DIR]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths:\n- ' + '\n- '.join(missing))

ip_to_id, id_to_ip = load_ip_map(RECOVERED_MAP)
scaler = joblib.load(SCALER_PATH)
relevant_flows, global_start, global_end = load_relevant_day2_flows(
    DAY2_CSV, pivot_ip=PIVOT_IP, attacker_ip=ATTACKER_IP
)
targets = select_targets(relevant_flows, ip_to_id, pivot_ip=PIVOT_IP)
audit = audit_ip_map_usage(
    DAY2_GRAPH_ROOT, ip_to_id, [PIVOT_IP, ATTACKER_IP, *targets['Target_IP']]
)
support = donor_support_report(relevant_flows)
attack_manifest = build_scenario_matrix(targets, global_start, global_end, pivot_ip=PIVOT_IP)
targets.to_csv(OUTPUT_DIR / 'selected_targets.csv', index=False)
support.to_csv(OUTPUT_DIR / 'donor_support_report.csv', index=False)
pd.DataFrame([audit]).to_csv(OUTPUT_DIR / 'ip_map_audit.csv', index=False)

print(audit)
display(targets)
display(support)
print('Attack scenarios:', len(attack_manifest))

## 2. Generate empirical-donor flows and sparse graph overlays

The original graph tensors remain the prefix of every modified graph. Only affected windows are written below each scenario directory.

In [ ]:
synthetic_attack_flows = create_synthetic_flows(
    relevant_flows, attack_manifest, global_start, attacker_ip=ATTACKER_IP, seed=GENERATION_SEED
)
control_flows, control_manifest = create_matched_controls(
    synthetic_attack_flows, attack_manifest, relevant_flows, ip_to_id
)
all_flows = pd.concat([synthetic_attack_flows, control_flows], ignore_index=True)
all_manifest = pd.concat([attack_manifest, control_manifest], ignore_index=True)
validation_issues = validate_synthetic_flows(all_flows, all_manifest, raise_on_error=False)
if not validation_issues.empty:
    display(validation_issues)
    raise SyntheticLMError('Synthetic-flow validation failed; overlays were not written.')

write_report_path = OUTPUT_DIR / 'overlay_write_report.csv'
if OVERWRITE_OVERLAYS or not write_report_path.exists():
    write_report = write_sparse_overlays(
        DAY2_GRAPH_ROOT, OUTPUT_DIR, all_flows, all_manifest, ip_to_id, scaler
    )
else:
    write_report = pd.read_csv(write_report_path)
    saved_flows = pd.read_csv(
        OUTPUT_DIR / 'synthetic_flows.csv',
        usecols=['Synthetic_Event_ID', 'Scenario_ID', 'Donor_Row_ID', 'window_id'],
    ).sort_values('Synthetic_Event_ID').reset_index(drop=True)
    expected_flows = all_flows[
        ['Synthetic_Event_ID', 'Scenario_ID', 'Donor_Row_ID', 'window_id']
    ].sort_values('Synthetic_Event_ID').reset_index(drop=True)
    pd.testing.assert_frame_equal(saved_flows, expected_flows, check_dtype=False)
    saved_scenarios = set(pd.read_csv(OUTPUT_DIR / 'scenario_manifest.csv')['scenario_id'])
    assert saved_scenarios == set(all_manifest['scenario_id']), 'Stored overlays are stale'

assert len(attack_manifest) == 54
assert len(control_manifest) == 9
assert write_report['Scenario_ID'].nunique() == 63
layout_report = validate_overlay_layout(OUTPUT_DIR, all_flows)
assert not layout_report[['Missing_Windows', 'Unexpected_Windows']].to_numpy().any()
display(write_report.groupby('Scenario_ID').agg(Windows=('Window_ID', 'nunique'), Synthetic_Flows=('Synthetic_Edges', 'sum')).head())

## 3. Load the unchanged champion checkpoints and thresholds

No synthetic result participates in champion selection or threshold selection.

In [ ]:
MODEL_CONFIG = {
    'node_dim': 16, 'edge_dim': 32, 'hidden_dim': 32,
    'dropout': 0.2, 'output_bias_init': -2.9968,
}
NAME_MAP = {
    'SimpleMLP_BiasOn': 'Simple MLP',
    'EGraphSAGE_BiasOn': 'E-GraphSAGE',
    'EdgeGRU_NoX_BiasOn': 'Edge GRU',
    'StaticGNN_BiasOn_robust_Identity': 'Static GNN',
    'ST_GNN_BiasOn_robust_Identity_clone': 'ST-GNN',
}
MODEL_SPECS = {
    'Simple MLP': (SimpleMLP, 'SimpleMLP_BiasOn', False),
    'E-GraphSAGE': (E_GraphSAGE, 'EGraphSAGE_BiasOn', False),
    'Edge GRU': (EdgeGRU_Baseline_NoX, 'EdgeGRU_NoX_BiasOn', True),
    'Static GNN': (StaticGNN_Identity, 'StaticGNN_BiasOn_robust_Identity', False),
    'ST-GNN': (ST_GNN_Identity, 'ST_GNN_BiasOn_robust_Identity_clone', True),
}

df_metrics = gather_metrics(RESULTS_DIR / 'logs', NAME_MAP)
df_champions = apply_1sd_rule(df_metrics)

def load_champion(model_class, experiment_name):
    champion = df_champions[df_champions['Raw_Dir_Name'].eq(experiment_name)]
    if champion.empty:
        raise ValueError(f'No champion found for {experiment_name}')
    seed = int(champion['Seed'].iloc[0])
    threshold_path = RESULTS_DIR / 'logs' / experiment_name / f'thresholds_{experiment_name}.npz'
    threshold = float(np.load(threshold_path)[f'seed_{seed}'])
    metric_candidates = [
        RESULTS_DIR / 'logs' / experiment_name / f'run_metrics_{experiment_name}.csv',
        RESULTS_DIR / 'logs' / experiment_name / f'metrics_newth_{experiment_name}.csv',
    ]
    metrics_path = next(path for path in metric_candidates if path.exists())
    metrics = pd.read_csv(metrics_path)
    metrics.columns = metrics.columns.str.lower()
    run_column = 'run_id' if 'run_id' in metrics.columns else 'extra_run_id'
    selected = metrics[metrics['model_name'].str.contains(f'seed{seed}', na=False)]
    run_id = selected[run_column].iloc[0]
    checkpoints = glob.glob(str(RESULTS_DIR / 'saved_models' / experiment_name / f'{run_id}_*.pth'))
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint for {experiment_name}, run {run_id}')
    model = model_class(**MODEL_CONFIG).to(DEVICE)
    model.load_state_dict(torch.load(checkpoints[0], map_location=DEVICE))
    model.eval()
    return model, threshold, seed

display(df_champions[df_champions['Raw_Dir_Name'].isin(NAME_MAP)][['Raw_Dir_Name', 'Seed', 'F1_Score']])

## 4. Fixed-model inference

Each model processes the base sequence once. Temporal memory is cached immediately before each scenario's first changed window, then restored for the short overlay interval.

In [ ]:
raw_timing = build_raw_timing_index(
    DAY2_CSV, [PIVOT_IP, *targets['Target_IP'].tolist()], global_start
)

all_base_events = []
all_scenario_predictions = []
for model_name, (model_class, experiment_name, temporal) in MODEL_SPECS.items():
    print(f'\nEvaluating {model_name}...')
    model, threshold, champion_seed = load_champion(model_class, experiment_name)
    base_events, predictions = evaluate_model_scenarios(
        model=model, model_name=model_name, threshold=threshold,
        base_graph_root=DAY2_GRAPH_ROOT, overlay_root=OUTPUT_DIR,
        scenario_manifest=all_manifest, id_to_ip=id_to_ip, pivot_ip=PIVOT_IP,
        device=DEVICE, temporal=temporal,
    )
    base_events = enrich_base_availability(base_events, raw_timing)
    base_events['Champion_Seed'] = champion_seed
    predictions['Champion_Seed'] = champion_seed
    base_events.to_csv(OUTPUT_DIR / f'base_events_{experiment_name}.csv', index=False)
    predictions.to_csv(OUTPUT_DIR / f'scenario_predictions_{experiment_name}.csv', index=False)
    all_base_events.append(base_events)
    all_scenario_predictions.append(predictions)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

base_events = pd.concat(all_base_events, ignore_index=True)
scenario_predictions = pd.concat(all_scenario_predictions, ignore_index=True)
base_events = attach_corrected_campaign_ground_truth(base_events, CORRECTED_CAMPAIGN_DIR)
base_events.to_csv(OUTPUT_DIR / 'base_events_all_models.csv', index=False)
scenario_predictions.to_csv(OUTPUT_DIR / 'scenario_predictions_all_models.csv', index=False)

## 5. Lead-time results and predeclared decision gate

A model passes only with ≥70% LM detection, ≥70% precursor coverage, median operational lead ≥5 minutes, coverage of at least two targets in two protocols, and ≥10 percentage points advantage over matched controls. The result remains a synthetic sensitivity finding.

In [ ]:
attack_summary, control_summary = summarize_experiment(
    scenario_predictions, base_events, all_manifest
)
gate = apply_diagnostic_gate(attack_summary, control_summary)
attack_summary.to_csv(OUTPUT_DIR / 'attack_scenario_summary.csv', index=False)
control_summary.to_csv(OUTPUT_DIR / 'matched_control_summary.csv', index=False)
gate.to_csv(OUTPUT_DIR / 'diagnostic_gate.csv', index=False)

display(gate)
if gate['Passes_Robust_Diagnostic_Gate'].any():
    print('CONTINUE: at least one fixed model passes the predeclared synthetic diagnostic gate.')
else:
    print('STOP: no fixed model passes the gate; move to a chain-labelled dataset.')

In [ ]:
# Compact result plots
sns.set_theme(style='whitegrid')
plot_data = attack_summary.groupby('Model', as_index=False).agg(
    LM_Coverage=('Synthetic_LM_Detected', 'mean'),
    Warning_Coverage=('Precursor_Warning', 'mean'),
    Median_Lead=('Operational_Lead_Minutes', 'median'),
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_data.set_index('Model')[['LM_Coverage', 'Warning_Coverage']].plot.bar(ax=axes[0], ylim=(0, 1))
sns.barplot(data=plot_data, x='Model', y='Median_Lead', ax=axes[1])
axes[0].set_title('Synthetic scenario coverage')
axes[1].set_title('Median operational precursor lead')
axes[1].tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'synthetic_lm_summary.png', dpi=180, bbox_inches='tight')
plt.show()

## Interpretation limits

- These are counterfactual aggregate flows assembled from empirical donor vectors.
- `Assumed_Success` is scenario ground truth, not success inferred from packet or host telemetry.
- A positive result shows that the fixed classifier reacts coherently to this constructed sequence.
- A negative result is a practical reason to proceed to a dataset containing captured, verified attack chains.
- Neither result estimates real-world lateral-movement forecasting performance.